<a href="https://colab.research.google.com/github/lobotomizedrequiem777-spec/sic-ia-clone-radar/blob/main/FairPlay_AI_DBSCAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FairPlay AI: detección de clones de videojuegos con DBSCAN

Ejercicio con **datos sintéticos** (descripciones + embeddings de imagen simulados).

- **Línea base:** TF-IDF de descripciones + DBSCAN
- **Modelo base:** distancia combinada (texto + imagen) + DBSCAN

> Un cluster indica similitud técnica, no plagio. La decisión final es de un especialista humano.

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

## 1. Dataset sintético
3 grupos de clones (*Escape the Walls*, *Farm Tycoon*, *Racing Legends*) y 13 juegos independientes, incluyendo señuelos del mismo género.

In [ ]:
rng = np.random.default_rng(42)
D_IMG = 32  # dimensión de los "embeddings CNN" simulados

grupos = {
    0: ("Escape the Walls",
        ["escapa de los muros que se cierran", "esquiva obstaculos",
         "sobrevive hasta llegar a la salida", "corre por pasillos que cambian"],
        ["Escape Walls Obby", "Walls Escape 3D", "Escape the Walls Clone", "Run From Walls",
         "Wall Runner Escape", "Escape the Wall Maze", "Walls Obby Escape"]),
    1: ("Farm Tycoon",
        ["construye tu granja", "cultiva y vende productos",
         "mejora tus herramientas", "contrata ayudantes"],
        ["Farm Tycoon Pro", "My Farm Empire", "Farm Tycoon Remake"]),
    2: ("Racing Legends",
        ["carreras de autos", "mejoras para tu vehiculo",
         "circuitos urbanos", "compite con otros jugadores"],
        ["Racing Legends 2", "Street Racing Legends"]),
}

# Juegos independientes (grupo -1). Incluye señuelos del mismo género que el grupo 0.
independientes = {
    "Obby Escape Tower": "sube una torre con saltos dificiles, evita lava y llega a la cima",
    "Escape the Lab": "resuelve acertijos en un laboratorio y encuentra la llave para salir",
    "Wall Climb Obby": "escapa de la lava que sube, esquiva obstaculos y sobrevive hasta la cima",
    "Escape the Prison": "escapa de una carcel, esquiva guardias y sobrevive hasta llegar a la salida",
    "Maze Runner Obby": "corre por pasillos de un laberinto, evita trampas y llega a la salida",
    "Pet Simulator Mini": "adopta mascotas, evoluciónalas y completa misiones diarias",
    "Zombie Siege": "defiende tu base de oleadas de zombies con armas mejoradas",
    "Cooking Chaos": "prepara pedidos rapidos en una cocina con tiempo limite",
    "Dino Park Builder": "construye un parque de dinosaurios y cuida a los visitantes",
    "Sky Parkour": "salta entre islas flotantes y desbloquea trucos de parkour",
    "Space Miner": "mina asteroides, vende minerales y mejora tu nave",
    "Hide and Seek Mansion": "escondete en una mansion embrujada mientras un monstruo busca",
    "Pizza Shop Tycoon": "administra tu pizzeria, atiende clientes y expande el local",
}

sinonimos = {"muros": "paredes", "obstaculos": "trampas", "granja": "finca",
             "autos": "coches", "vehiculo": "carro", "escapa": "huye",
             "esquiva": "evita", "sobrevive": "aguanta", "construye": "arma",
             "carreras": "competencias"}

def variar(frases):
    """Simula una descripción reescrita (pocas frases + sinónimos)."""
    f = list(frases)
    rng.shuffle(f)
    f = f[:rng.integers(1, 3)]
    txt = ", ".join(f)
    for a, b in sinonimos.items():
        if rng.random() < 0.6:
            txt = txt.replace(a, b)
    return txt

filas, centros = [], {}
for g, (orig, frases, clones) in grupos.items():
    centros[g] = rng.normal(0, 1, D_IMG)  # "estilo visual" común del grupo
    for nombre in [orig] + clones:
        emb = centros[g] + rng.normal(0, 1.1, D_IMG)  # capturas con ruido
        filas.append((nombre, variar(frases), g, emb))
for nombre, desc in independientes.items():
    filas.append((nombre, desc, -1, rng.normal(0, 1, D_IMG)))

df = pd.DataFrame({"id": range(len(filas)),
                   "titulo": [f[0] for f in filas],
                   "descripcion": [f[1] for f in filas],
                   "grupo_real": [f[2] for f in filas]})
emb = np.array([f[3] for f in filas])
y = df["grupo_real"].values

print("Juegos por grupo real (-1 = sin clones):")
print(df["grupo_real"].value_counts().sort_index(), "\n")

# (Opcional) guardar el dataset como CSV
# pd.concat([df, pd.DataFrame(emb, columns=[f"img_{i}" for i in range(D_IMG)]).round(4)], axis=1)\
#   .to_csv("juegos_sinteticos.csv", index=False)

Juegos por grupo real (-1 = sin clones):
grupo_real
-1    13
 0     8
 1     4
 2     3
Name: count, dtype: int64 



## 2. Matrices de distancia
Coseno sobre TF-IDF (texto), sobre embeddings (imagen) y su promedio.

In [ ]:
D_texto = cosine_distances(TfidfVectorizer().fit_transform(df["descripcion"]))  # línea base
D_img = cosine_distances(emb)                                                    # capturas (CNN)
D_comb = 0.5 * D_texto + 0.5 * D_img                                             # modelo base

## 3. Evaluación por pares y barrido de `eps`
Precisión y recall sobre pares de juegos que el modelo agrupa como clones.

In [ ]:
def pares_clon(etiquetas):
    return {(i, j) for i, j in combinations(range(len(etiquetas)), 2)
            if etiquetas[i] == etiquetas[j] and etiquetas[i] != -1}

def evaluar(nombre, D, eps, min_samples=2):
    labels = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit_predict(D)
    real, pred = pares_clon(y), pares_clon(labels)
    tp = len(real & pred)
    prec = tp / len(pred) if pred else 0
    rec = tp / len(real) if real else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    met = dict(modelo=nombre, eps=round(float(eps), 2), precision=prec, recall=rec,
               F1=f1, ARI=adjusted_rand_score(y, labels), ruido=int((labels == -1).sum()))
    return met, labels

def mejor(nombre, D):
    return max((evaluar(nombre, D, e) for e in np.arange(0.2, 1.0, 0.05)),
               key=lambda t: t[0]["F1"])

(m_base, lab_base) = mejor("Linea base (TF-IDF)", D_texto)
(m_mod, lab_mod) = mejor("Modelo base (texto+imagen)", D_comb)

print("=== MODELO BASE vs LINEA BASE ===")
print(pd.DataFrame([m_base, m_mod]).round(3).to_string(index=False), "\n")

=== MODELO BASE vs LINEA BASE ===
                    modelo  eps  precision  recall    F1   ARI  ruido
       Linea base (TF-IDF) 0.75      0.479   0.946 0.636 0.510     10
Modelo base (texto+imagen) 0.60      1.000   0.838 0.912 0.883     14 



## 4. Tabla de 3 casos probados
Clon evidente, mismo género y sin relación.

In [ ]:
idx = lambda n: int(df.index[df.titulo == n][0])

def caso(i, j, nombre):
    return {"caso": nombre, "par": f"{df.titulo[i]} vs {df.titulo[j]}",
            "sim_texto": round(1 - D_texto[i, j], 2),
            "sim_imagen": round(1 - D_img[i, j], 2),
            "sim_combinada": round(1 - D_comb[i, j], 2),
            "mismo_cluster_base": bool(lab_base[i] == lab_base[j] and lab_base[i] != -1),
            "mismo_cluster_modelo": bool(lab_mod[i] == lab_mod[j] and lab_mod[i] != -1)}

tabla = pd.DataFrame([
    caso(idx("Escape the Walls"), idx("Escape Walls Obby"), "1 Clon evidente"),
    caso(idx("Escape the Walls"), idx("Escape the Lab"), "2 Mismo genero"),
    caso(idx("Escape the Walls"), idx("Farm Tycoon"), "3 Sin relacion"),
])
print("=== 3 CASOS PROBADOS ===")
print(tabla.to_string(index=False))
print("\nNota: un cluster indica similitud tecnica, no plagio. La decision final es de un especialista humano.")

=== 3 CASOS PROBADOS ===
           caso                                   par  sim_texto  sim_imagen  sim_combinada  mismo_cluster_base  mismo_cluster_modelo
1 Clon evidente Escape the Walls vs Escape Walls Obby        0.0        0.52           0.26                True                  True
 2 Mismo genero    Escape the Walls vs Escape the Lab        0.0       -0.04          -0.02               False                 False
 3 Sin relacion       Escape the Walls vs Farm Tycoon        0.0        0.33           0.16               False                 False

Nota: un cluster indica similitud tecnica, no plagio. La decision final es de un especialista humano.
